# Keyword Analysis — Classi problematiche Area Classifier

Obiettivo: trovare i termini più discriminanti per le 5 classi con F1 più basso,
usando TF-IDF come rappresentazione e chi-quadro come misura di associazione.

**Output**: tabella `keyword_analysis_area.csv` da ispezionare manualmente.
Dopo l'ispezione, i termini selezionati diventeranno colonne booleane in `dataset_clean.csv`.

| Classe | F1 attuale |
|---|---|
| `area_territoriale` | 0.37 |
| `sistema381` | 0.45 |
| `area_sistemistica` | 0.53 |
| `ciclo_attivo` | 0.67 |
| `ciclo_passivo` | 0.72 |

---
## STEP 1 — Import e caricamento dati

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

# notebooks/analysis/ → .parent → notebooks/ → .parent → TicketClassifier/
BASE_DIR = Path(os.path.abspath('')).parent.parent
CSV_PATH = BASE_DIR / 'data' / 'dataset_clean.csv'
OUT_PATH = BASE_DIR / 'data' / 'keyword_analysis_area.csv'

df = pd.read_csv(CSV_PATH, parse_dates=['data_creazione'])
print(f'Righe caricate: {len(df):,}')
print(f'Colonne: {df.columns.tolist()}')

Righe caricate: 58,385
Colonne: ['url_ticket', 'case_number', 'testo_input', 'priorita_finale', 'priorita_iniziale_cliente', 'area', 'articolo', 'modulo_sw', 'has_urgenza', 'n_parole', 'data_creazione', 'kw_s381_codice', 'kw_s381_rapportino', 'kw_s381_timbrate', 'kw_s381_calendario_presenze', 'kw_ter_unodomo', 'kw_ter_distretto', 'kw_pas_iva', 'kw_pas_fornitore', 'kw_pas_cespiti', 'kw_pas_prima_nota', 'kw_pas_ammortamento', 'kw_pas_analitica', 'kw_pas_reverse', 'kw_att_retta', 'kw_att_pagopa', 'kw_att_sdd', 'kw_att_portale_utenti', 'kw_att_fattura_elettronica']


---
## STEP 2 — Pulizia classi (stessa logica di AreaClassifier)

In [2]:
# Replica esatta della pulizia fatta in AreaClassifier_VSCode.ipynb
df['area_v2'] = df['area'].replace({'Hardware': 'area_sistemistica'})

CLASSI_DROP = ['business_intelligence', 'protocollo_delibere']
df.loc[df['area_v2'].isin(CLASSI_DROP), 'area_v2'] = np.nan

conteggi = df['area_v2'].value_counts()
classi_rumore = conteggi[conteggi < 10].index.tolist()
df.loc[df['area_v2'].isin(classi_rumore), 'area_v2'] = np.nan

# Teniamo solo i ticket con area valorizzata e testo non vuoto
df_validi = df.dropna(subset=['area_v2', 'testo_input']).copy()
df_validi['testo_lower'] = df_validi['testo_input'].str.lower()

print(f'Ticket usabili: {len(df_validi):,}')
print()
print('Distribuzione classi:')
print(df_validi['area_v2'].value_counts().to_string())

Ticket usabili: 52,149

Distribuzione classi:
area_v2
area_personale                     14251
ciclo_passivo                      10991
ciclo_attivo                        9233
area_sanitaria                      8747
rendicontazione_flussi              2854
protocollo_documentale_delibere     2580
area_sistemistica                   1951
sistema381                          1006
area_territoriale                    536


---
## STEP 3 — TF-IDF sul corpus completo

Costruiamo la matrice TF-IDF una volta sola su tutti i ticket validi.
Usiamo unigrammi e bigrammi con un minimo di 20 occorrenze per filtrare il rumore.

In [3]:
# TF-IDF: unigrammi + bigrammi, almeno 20 doc, max 10k features
# sublinear_tf=True riduce il peso dei termini molto frequenti (log scaling)
vec = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=20,
    max_features=10_000,
    sublinear_tf=True
)

X_tfidf = vec.fit_transform(df_validi['testo_lower'])
termini = vec.get_feature_names_out()

print(f'Matrice TF-IDF: {X_tfidf.shape}  ({X_tfidf.shape[1]} termini)')

Matrice TF-IDF: (52149, 10000)  (10000 termini)


---
## STEP 4 — Chi-quadro per le 5 classi problematiche

Per ogni classe facciamo un test chi-quadro **one-vs-rest**:
- variabile dipendente: 1 se il ticket appartiene alla classe, 0 altrimenti
- variabile indipendente: frequenza del termine (TF-IDF)

Il chi-quadro misura quanto la presenza di un termine è statisticamente
associata all'appartenenza alla classe, indipendentemente dalla frequenza assoluta.

In [4]:
# Classi su cui vogliamo le keyword (quelle con F1 basso)
CLASSI_TARGET = [
    'area_territoriale',  # F1 0.37
    'sistema381',         # F1 0.45
    'area_sistemistica',  # F1 0.53
    'ciclo_attivo',       # F1 0.67
    'ciclo_passivo',      # F1 0.72
]

TOP_N = 30  # top termini da estrarre per classe

righe = []  # raccoglieremo qui i risultati

for classe in CLASSI_TARGET:
    # Etichetta binaria: 1 = questa classe, 0 = tutto il resto
    y_bin = (df_validi['area_v2'] == classe).astype(int).values
    n_pos = y_bin.sum()

    # Chi-quadro: restituisce score e p-value per ogni termine
    chi_scores, pvals = chi2(X_tfidf, y_bin)

    # Prendiamo i TOP_N termini con chi2 più alto
    top_idx = chi_scores.argsort()[::-1][:TOP_N]

    for rank, idx in enumerate(top_idx, start=1):
        termine = termini[idx]
        score   = chi_scores[idx]
        pval    = pvals[idx]

        # Precision del termine: quanti ticket con questa parola sono davvero della classe?
        # Serve per capire se la keyword è esclusiva o condivisa con altre classi
        mask_termine = np.asarray(X_tfidf[:, idx].todense()).flatten() > 0
        n_con_termine = mask_termine.sum()
        n_classe_con_termine = (mask_termine & (y_bin == 1)).sum()
        precision = n_classe_con_termine / n_con_termine if n_con_termine > 0 else 0

        # Recall: quanti ticket della classe contengono questo termine?
        recall = n_classe_con_termine / n_pos if n_pos > 0 else 0

        righe.append({
            'classe':         classe,
            'rank':           rank,
            'termine':        termine,
            'chi2':           round(score, 1),
            'pval':           round(pval, 6),
            'precision':      round(precision, 3),  # % ticket con il termine che sono di questa classe
            'recall':         round(recall, 3),     # % ticket della classe che contengono il termine
            'n_occorrenze':   int(n_con_termine),
            'n_classe':       int(n_pos),
        })

    print(f'{classe:<35} ({n_pos:,} ticket) — analisi completata')

df_kw = pd.DataFrame(righe)
print(f'\nTabella risultati: {len(df_kw)} righe')

area_territoriale                   (536 ticket) — analisi completata
sistema381                          (1,006 ticket) — analisi completata
area_sistemistica                   (1,951 ticket) — analisi completata
ciclo_attivo                        (9,233 ticket) — analisi completata
ciclo_passivo                       (10,991 ticket) — analisi completata

Tabella risultati: 150 righe


---
## STEP 5 — Visualizzazione risultati

Per ogni classe mostriamo i top 20 termini ordinati per chi2.

**Come leggere la tabella:**
- `chi2`: quanto il termine è statisticamente associato alla classe (più alto = più discriminante)
- `precision`: % di ticket *con quel termine* che appartengono a questa classe (alta = termine esclusivo)
- `recall`: % di ticket *di questa classe* che contengono il termine (alta = termine diffuso nella classe)
- `n_occorrenze`: quante volte appare nel corpus totale

**Cosa tenere**: termini con precision alta (>0.70) e recall decente (>0.10), escludendo boilerplate.

**Cosa scartare**: termini generici (`errore`, `problema`, `sistema`), nomi propri di persone, boilerplate email.

In [5]:
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

for classe in CLASSI_TARGET:
    subset = df_kw[df_kw['classe'] == classe].head(20)
    print(f'\n{"="*70}')
    print(f'  {classe.upper()}  —  {subset["n_classe"].iloc[0]:,} ticket')
    print(f'{"="*70}')
    print(subset[['rank', 'termine', 'chi2', 'precision', 'recall', 'n_occorrenze']].to_string(index=False))


  AREA_TERRITORIALE  —  536 ticket
 rank        termine     chi2  precision  recall  n_occorrenze
    1        unodomo 1214.800      0.928   0.119            69
    2           domo 1068.400      0.894   0.110            66
    3       uno domo 1056.000      0.932   0.103            59
    4            sad  845.800      0.413   0.207           269
    5     interventi  602.900      0.458   0.132           155
    6        accessi  561.600      0.356   0.164           247
    7      distretto  369.100      0.562   0.050            48
    8             uo  279.900      0.417   0.065            84
    9    sospensione  215.600      0.217   0.047           115
   10    sospensioni  196.900      0.455   0.028            33
   11 programmazione  155.600      0.170   0.086           270
   12       progetti  148.300      0.210   0.041           105
   13          giada  139.400      0.308   0.037            65
   14    diagnostica  138.700      0.340   0.032            50
   15        del sa

---
## STEP 6 — Salvataggio tabella per ispezione manuale

In [6]:
df_kw.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')  # utf-8-sig per aprire bene in Excel
print(f'Salvato: {OUT_PATH}')
print()
print('Prossimo step: ispeziona keyword_analysis_area.csv e segnala quali termini')
print('vuoi trasformare in colonne booleane in dataset_clean.csv')

Salvato: c:\Users\matteo.segatto\Desktop\TicketClassifier\data\keyword_analysis_area.csv

Prossimo step: ispeziona keyword_analysis_area.csv e segnala quali termini
vuoi trasformare in colonne booleane in dataset_clean.csv
